# 32 - Failed matcher29: crop and identity-signal audit (READ ONLY)

**No training, optimizer, NBIS, new weights or GPU operations.** Uses completed29 even though its final cell
raised QUALITY GATE FAILED:29 published the checksummed artifacts BEFORE intentionally raising.
After a runtime reset, restore source images with02 on the SAME runtime;04 is not required.
CPU inference is forced even in a GPU Colab session. Keep the same frozen library versions.
Do not pay for A100 for this audit; a new runtime must still satisfy the version checks.

1. Verify completed29 pointer, all file hashes, fixed-budget LAST checkpoint, exact original train-only partition,
   score pair metadata and replay saved gate from CSV with round-trip float parsing.
2. Predefined balanced panels: original gate groups and the first eight FIT connected groups per DB,
   picked by fixed hash (not scores), with one gallery and two byte-distinct impressions per identity.
3. Use the SAME frozen matcher on CPU: same-source original/degraded features versus different-impression
   genuine features; same-DB versus cross-DB impostors. Fit-panel performance is in-sample descriptive.
   Cross-DB comparisons are a diagnostic negative pool, NOT an operational FMR or a revised quality gate.
4. Reproduce the exact original foreground-anchored192px crop with its path-dependent RNG. Save its geometry,
   original-coordinate crop fraction and previews. Crop rectangles in different impressions are NOT registered;
   coordinate displacement/area alone cannot establish anatomical overlap.

Saved29 GPU gate scores are the authoritative gate evidence. CPU re-inference may differ numerically:
report max score error and decision flips, never overwrite the gate or reinterpret CPU results as a new pass.
Use original calibration thresholds throughout; no calibration, model update, gate relaxation, seed reroll,
validation/external/test image access or write under29. No automatic causal claim, repair or unlock of30/31.
Writes only a new `diagnostics/risk_matcher_failure_v1/<hash>/` directory on Drive.

Interpretation: good same-source but poor cross-impression scores support a mismatch between aligned-patch
restoration supervision and identity matching; good FIT but poor gate suggests a seen/unseen gap.
Much easier cross-DB than same-DB negatives are consistent with database cues, NOT proof of a sensor shortcut.
These observations do not isolate capacity, optimization, geometric overlap or fingerprint biology.


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import inspect
import json
import math
import shutil
import sys
import time
import types
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Subset
from google.colab import drive
DEVICE = torch.device('cpu')
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
M0_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
M1_DIR = PROJECT_ROOT / 'experiments' / 'm1_shared_curriculum_v1_seed20261110' / 'dd1427cfad6621c577bf'
SEED = 20261110
RUN_SEEDS = {'dry_l1_seed20261110': 20261110}
RUN_NAMES = tuple(RUN_SEEDS)
M2_DIR = PROJECT_ROOT / 'experiments' / 'm2_structure_loss_v1_seed20261110' / '92579202157d85254b7f'
SPECIALIZATION_ROOT = PROJECT_ROOT / 'experiments' / 'm5_dry_specialization_v1'
LAMBDA_GRADIENT = 0.1
IDENTITY_FRACTION = 0.1

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

def verify_files(directory, hashes):
    for name, expected in hashes.items():
        if file_digest(directory / name) != expected:
            raise RuntimeError(f'Frozen artifact changed: {directory / name}')
print('CPU-only read-only matcher audit; no training, GPU operations or NBIS.')
MATCHER_ROOT = PROJECT_ROOT / 'experiments' / 'risk_patch_matcher_v1'
MATCHER_SETTINGS = {'epochs': 6, 'steps_per_epoch': 200, 'identities_per_batch': 12,
    'learning_rate': 0.0002, 'embedding_dim': 128, 'held_groups_per_db': 8,
    'quality_gate': {'clean_auc': 0.80, 'dry_auc': 0.70, 'clean_tar': 0.25, 'dry_tar': 0.10}}


In [ ]:
split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
verify_files(SPLIT_DIR, split_protocol['file_sha256'])
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
m0_config = json.loads((M0_DIR / 'configuration.json').read_text(encoding='utf-8'))
m1_config = json.loads((M1_DIR / 'configuration.json').read_text(encoding='utf-8'))
m1_completion = json.loads((M1_DIR / 'completion.json').read_text(encoding='utf-8'))
sim_config = json.loads((SIM_DIR / 'configuration.json').read_text(encoding='utf-8'))
pair_config = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))
m0_sha, m1_sha = (file_digest(directory / 'configuration.json') for directory in (M0_DIR, M1_DIR))
verify_files(M0_DIR, m0_config['source_file_sha256'])
verify_files(M1_DIR, m1_config['source_file_sha256'])
verify_files(M1_DIR, m1_completion['file_sha256'])
if (m1_completion['status'] != 'shared_curriculum_control_complete' or m1_completion['configuration_sha256'] != m1_sha
        or m1_config['m0_configuration_sha256'] != m0_sha):
    raise RuntimeError('Frozen completed M1 provenance mismatch.')
if any(config['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json') for config in (m0_config, sim_config, pair_config)):
    raise RuntimeError('Split provenance differs across notebooks.')
if (m0_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or m1_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or sim_config['normalization_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or file_digest(SIM_DIR / 'fingerprint_degradation.py') != sim_config['module_sha256']
        or m0_config['simulator_module_sha256'] != sim_config['module_sha256']):
    raise RuntimeError('Frozen simulator/normalization provenance mismatch.')
for config in (m0_config, m1_config):
    if (str(torch.__version__) != config['torch_version'] or str(torch.version.cuda) != config['torch_cuda']
            or np.__version__ != config['numpy_version'] or Image.__version__ != config['pillow_version']):
        raise RuntimeError('Library versions differ from frozen training.')
if Image.__version__ != pair_config['pillow_version'] or Image.__version__ != sim_config['pillow_version'] or np.__version__ != sim_config['numpy_version']:
    raise RuntimeError('Normalization/simulator library versions differ.')
for name in ('restoration_baseline.py', 'train_sources.csv', 'validation_sources.csv'):
    if file_digest(M1_DIR / name) != file_digest(M0_DIR / name):
        raise RuntimeError(f'M1 model/source differs from M0: {name}')
SETTINGS = dict(m1_config['settings'])
EPOCHS = len(m1_config['epoch_conditions'])
if (EPOCHS != sum(SETTINGS['phase_lengths']) or EPOCHS != m0_config['epochs']
        or SETTINGS['base_channels'] != m0_config['base_channels'] or SETTINGS['patch_size'] != m0_config['patch_size']
        or SETTINGS['batch_size'] != m0_config['batch_size'] or SETTINGS['learning_rate'] != m0_config['learning_rate']):
    raise RuntimeError('Budget/architecture/optimizer differ from the frozen baseline.')
train_sources = pd.read_csv(M0_DIR / 'train_sources.csv', keep_default_na=False)
validation_sources = pd.read_csv(M0_DIR / 'validation_sources.csv', keep_default_na=False)
for split_name, frame in (('train', train_sources), ('validation', validation_sources)):
    allowed = locked[locked['split'].eq(split_name)].set_index('relative_path')
    if frame.empty or frame['relative_path'].duplicated().any() or frame.groupby(['dataset', 'db']).ngroups != 6 or set(frame['dataset']) != {'FVC2000', 'FVC2002'}:
        raise RuntimeError(f'Invalid source-only {split_name} data.')
    for record in frame.to_dict('records'):
        relative = record['relative_path']
        if (relative not in allowed.index or allowed.loc[relative, 'sha256'] != record['sha256']
                or allowed.loc[relative, 'identity_key'] != record['identity_key']):
            raise RuntimeError(f'Split/source mismatch: {relative}')
        path = DATA_ROOT / relative
        if split_name == 'train' and (not path.is_file() or file_digest(path) != record['sha256']):
            raise RuntimeError(f'Image missing/changed: {path}. Restore with notebook 02.')
if (set(train_sources['identity_key']) & set(validation_sources['identity_key'])
        or set(train_sources['sha256']) & set(validation_sources['sha256'])
        or not validation_sources.groupby(['dataset', 'db']).size().eq(12).all()):
    raise RuntimeError('Train/validation overlap or changed validation coverage.')
training = load_module('m2_original_model_v1', M0_DIR / 'restoration_baseline.py')
simulator = load_module('m2_original_simulator_v1', SIM_DIR / 'fingerprint_degradation.py')
curriculum_data = load_module('m2_frozen_task_data_v1', M1_DIR / 'curriculum_data.py')
FROZEN_ENGINE_SOURCE = (M1_DIR / 'training_helpers.py.txt').read_bytes().decode('utf-8')
SOURCE_COUNT = len(train_sources)
all_conditions = [('identity', 'none')] + [(family, severity)
    for family in ('dry_like', 'wet_like', 'generic') for severity in ('mild', 'moderate', 'severe')]
print('Verified unchanged sources:', SOURCE_COUNT, 'train /', len(validation_sources), 'validation.')

allowed_train_paths = frozenset(train_sources.relative_path)


In [ ]:

pointer = json.loads((MATCHER_ROOT / 'completed_experiment.json').read_text(encoding='utf-8'))
MATCHER_DIR = Path(pointer['directory'])
if (MATCHER_DIR.parent != MATCHER_ROOT
        or file_digest(MATCHER_DIR / 'configuration.json') != pointer['configuration_sha256']
        or file_digest(MATCHER_DIR / 'completion.json') != pointer['completion_sha256']):
    raise RuntimeError('Notebook29 published pointer/integrity mismatch.')
configuration = json.loads((MATCHER_DIR / 'configuration.json').read_text(encoding='utf-8'))
completion = json.loads((MATCHER_DIR / 'completion.json').read_text(encoding='utf-8'))
if (configuration['version'] != 'risk_patch_matcher_v1'
        or configuration['settings'] != MATCHER_SETTINGS or configuration['seed'] != SEED
        or completion['status'] != 'risk_matcher_fit_complete'
        or completion['configuration_sha256'] != pointer['configuration_sha256']
        or completion['completed_epochs'] != MATCHER_SETTINGS['epochs']
        or configuration['selection'] != 'last fixed-budget checkpoint; no gate/validation epoch selection'
        or configuration['train_sources_sha256'] != file_digest(M0_DIR / 'train_sources.csv')
        or configuration['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json')
        or configuration['simulator_sha256'] != file_digest(SIM_DIR / 'fingerprint_degradation.py')
        or configuration['patch_size'] != SETTINGS['patch_size']):
    raise RuntimeError('Expected unchanged completed29 fixed-budget matcher configuration.')
verify_files(MATCHER_DIR, completion['file_sha256'])
for key, actual in (('torch_version', str(torch.__version__)), ('torch_cuda', str(torch.version.cuda)),
        ('numpy_version', np.__version__), ('pillow_version', Image.__version__)):
    if configuration[key] != actual:
        raise RuntimeError(f'Matcher library version differs: {key}')
matcher_source = (MATCHER_DIR / 'matcher_source.py').read_bytes().decode('utf-8')
if (hashlib.sha256(matcher_source.encode()).hexdigest() != configuration['model_source_sha256']
        or file_digest(MATCHER_DIR / 'inner_partition.csv') != configuration['partition_sha256']
        or file_digest(MATCHER_DIR / 'checkpoint_helpers.py.txt') != configuration['checkpoint_source_sha256']):
    raise RuntimeError('Frozen matcher source/partition hash differs.')
matcher = types.ModuleType('read_only_failed29_matcher')
matcher.__dict__.update(globals())
exec(compile(matcher_source, str(MATCHER_DIR / 'matcher_source.py'), 'exec'), matcher.__dict__)
partition = pd.read_csv(MATCHER_DIR / 'inner_partition.csv', keep_default_na=False)
expected_partition = matcher.inner_partition(train_sources, MATCHER_SETTINGS['held_groups_per_db'])
pd.testing.assert_frame_equal(partition, expected_partition, check_dtype=False)
latest = json.loads((MATCHER_DIR / 'latest.json').read_text(encoding='utf-8'))
if (latest != completion['checkpoint'] or Path(latest['file']).name != latest['file']
        or latest['configuration_sha256'] != pointer['configuration_sha256']
        or latest['completed_epochs'] != MATCHER_SETTINGS['epochs']
        or file_digest(MATCHER_DIR / latest['file']) != latest['sha256']):
    raise RuntimeError('LAST completed29 checkpoint differs.')
state = torch.load(MATCHER_DIR / latest['file'], map_location='cpu', weights_only=True)
if (state['configuration_sha256'] != pointer['configuration_sha256']
        or state['completed_epochs'] != MATCHER_SETTINGS['epochs']):
    raise RuntimeError('Saved matcher payload differs.')
with torch.random.fork_rng(devices=[]):
    model = matcher.PatchMatcher().to(DEVICE)
model.load_state_dict(state['model'], strict=True)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
if any(not torch.isfinite(value).all() for value in model.state_dict().values()):
    raise RuntimeError('Nonfinite frozen matcher state.')
saved_state = {key: value.clone() for key, value in model.state_dict().items()}
saved_history = pd.DataFrame(state['history'])
if saved_history.epoch.tolist() != list(range(1, MATCHER_SETTINGS['epochs'] + 1)):
    raise RuntimeError('Matcher has not trained the full predefined budget.')
del state
calibration = pd.read_csv(MATCHER_DIR / 'calibration_scores.csv', float_precision='round_trip')
gate = pd.read_csv(MATCHER_DIR / 'gate_scores.csv', float_precision='round_trip')
saved_gate = json.loads((MATCHER_DIR / 'quality_gate.json').read_text(encoding='utf-8'))
saved_thresholds = json.loads((MATCHER_DIR / 'thresholds.json').read_text(encoding='utf-8'))
if saved_gate != completion['quality_gate']:
    raise RuntimeError('Saved29 gate attestation differs.')
print('Read-only loaded matcher epoch:', latest['completed_epochs'], '| original gate passed:', saved_gate['passed'])


In [ ]:
AUDIT_SOURCE = 'def panel_records(partition, split):\n    if split not in (\'fit\', \'gate\', \'calibration\'):\n        raise ValueError(\'Only explicit main-TRAIN inner partitions are supported.\')\n    selected = partition[partition.matcher_split.eq(split)]\n    if selected.empty:\n        raise RuntimeError(\'Missing inner partition.\')\n    if split == \'fit\':\n        frames = []\n        for (dataset, db), rows in selected.groupby([\'dataset\', \'db\'], sort=True):\n            groups = sorted(rows.connected_group.unique(),\n                key=lambda key: training.stable_seed(f\'{SEED}|audit_fit_panel|{key}\'))\n            if len(groups) < MATCHER_SETTINGS[\'held_groups_per_db\']:\n                raise RuntimeError(\'Insufficient fit groups for matched audit panel.\')\n            frames.append(rows[rows.connected_group.isin(groups[:MATCHER_SETTINGS[\'held_groups_per_db\']])])\n        selected = pd.concat(frames, ignore_index=True)\n    galleries, probes = [], []\n    for key, rows in selected.groupby(\'identity_key\', sort=True):\n        records = rows.sort_values(\'relative_path\').drop_duplicates(\'sha256\').to_dict(\'records\')\n        if len(records) < 3:\n            raise RuntimeError(f\'Audit requires the same three byte-distinct source impressions: {key}\')\n        galleries.append(records[0])\n        probes.extend(records[1:3])\n    return galleries, probes\n\ndef pair_metadata(partition, split):\n    galleries, probes = panel_records(partition, split)\n    rows = []\n    for condition in (\'identity/none\', \'dry_like/mild\', \'dry_like/moderate\', \'dry_like/severe\'):\n        for probe in probes:\n            for gallery in galleries:\n                if (probe[\'dataset\'], probe[\'db\']) != (gallery[\'dataset\'], gallery[\'db\']):\n                    continue\n                label = int(probe[\'identity_key\'] == gallery[\'identity_key\'])\n                if not label and probe[\'connected_group\'] == gallery[\'connected_group\']:\n                    continue\n                rows.append(dict(dataset=probe[\'dataset\'], db=probe[\'db\'], condition=condition,\n                    probe=probe[\'relative_path\'], gallery=gallery[\'relative_path\'], label=label))\n    return pd.DataFrame(rows)\n\ndef verify_score_metadata(frame, partition, split):\n    keys = [\'dataset\', \'db\', \'condition\', \'probe\', \'gallery\', \'label\']\n    if (frame.empty or frame.duplicated(keys).any() or not np.isfinite(frame.score).all()\n            or not frame.score.between(-1.00001, 1.00001).all() or set(frame.label) != {0, 1}):\n        raise RuntimeError(\'Invalid saved inner verification scores.\')\n    expected = pair_metadata(partition, split)\n    pd.testing.assert_frame_equal(frame[keys].sort_values(keys).reset_index(drop=True),\n        expected[keys].sort_values(keys).reset_index(drop=True), check_dtype=False)\n\ndef crop_details(record):\n    relative = record[\'relative_path\']\n    if relative not in allowed_train_paths:\n        raise RuntimeError(\'Crop audit can read main TRAIN images only.\')\n    dpi = pair_config[\'source_dpi\'][f"{record[\'dataset\']}_{record[\'db\']}"]\n    reference = training.normalized_image(str(DATA_ROOT / relative), dpi)\n    foreground = simulator.foreground_mask(reference)\n    ys, xs = np.where(foreground)\n    if not len(ys):\n        raise RuntimeError(f\'No foreground for frozen crop: {relative}\')\n    crop_seed = training.stable_seed(f\'{SEED}|0|{relative}|crop\')\n    rng = np.random.default_rng(crop_seed)\n    anchor = int(rng.integers(len(ys)))\n    size = SETTINGS[\'patch_size\']\n    height, width = reference.shape\n    top = int(np.clip(int(ys[anchor]) - size // 2, 0, max(0, height - size)))\n    left = int(np.clip(int(xs[anchor]) - size // 2, 0, max(0, width - size)))\n    pad = ((0, max(0, size - height)), (0, max(0, size - width)))\n    padded = np.pad(reference, pad, mode=\'reflect\') if any(value for pair in pad for value in pair) else reference\n    patch = np.array(padded[top:top + size, left:left + size], dtype=np.float32, copy=True)\n    actual = matcher.sample_patches([record], 0, [(\'identity\', \'none\')])[0, 0].numpy()\n    if not np.array_equal(patch, actual):\n        raise RuntimeError(\'Reconstructed crop is not byte-identical to frozen29 sampling.\')\n    coverage = (min(size, height - top) * min(size, width - left)) / (height * width)\n    metadata = dict(relative_path=relative, identity_key=record[\'identity_key\'], dataset=record[\'dataset\'],\n        db=record[\'db\'], crop_seed=crop_seed, normalized_height=height, normalized_width=width,\n        top=top, left=left, patch_size=size, padded_bottom=pad[0][1], padded_right=pad[1][1],\n        crop_image_fraction=coverage, crop_foreground_fraction=float(\n            foreground[top:min(top + size, height), left:min(left + size, width)].mean()),\n        crop_sha256=hashlib.sha256(patch.astype(\'<f4\').tobytes()).hexdigest())\n    return metadata, reference, patch\n\ndef describe_scores(values):\n    values = np.asarray(values, dtype=np.float64)\n    if not len(values) or not np.isfinite(values).all():\n        raise RuntimeError(\'Nonempty finite scores required for descriptive summaries.\')\n    return dict(count=len(values), mean=float(values.mean()), std=float(values.std(ddof=1)) if len(values) > 1 else None,\n        q05=float(np.quantile(values, .05)), median=float(np.median(values)), q95=float(np.quantile(values, .95)))\n\ndef auc_against(genuine, impostors):\n    genuine, impostors = np.asarray(genuine), np.asarray(impostors)\n    if not len(genuine) or not len(impostors):\n        raise RuntimeError(\'Both positive and negative scores required.\')\n    return float(np.mean((genuine[:, None] > impostors).astype(float)\n        + .5 * (genuine[:, None] == impostors)))\n\n@torch.inference_mode()\ndef embed_panel(records, condition):\n    features = []\n    for start in range(0, len(records), 12):\n        images = matcher.sample_patches(records[start:start + 12], 0, [condition]).to(DEVICE)\n        value = model(images)\n        if not torch.isfinite(value).all():\n            raise RuntimeError(\'Nonfinite CPU embeddings.\')\n        features.append(value.cpu())\n    return torch.cat(features)\n\ndef summarize_panel(frame):\n    summaries = []\n    for (split, dataset, db, condition), group in frame.groupby(\n            [\'inner_split\', \'dataset\', \'db\', \'condition\'], sort=True):\n        threshold = saved_thresholds[f\'{dataset}_{db}\']\n        positive = group[group.pool.eq(\'different_impression_genuine\')].score.to_numpy()\n        for pool in (\'different_impression_genuine\', \'same_source_control\', \'same_db_impostor\', \'cross_db_impostor\'):\n            values = group[group.pool.eq(pool)].score.to_numpy()\n            summaries.append(dict(inner_split=split, dataset=dataset, db=db, condition=condition, pool=pool,\n                threshold=threshold, acceptance_fraction=float(np.mean(values >= threshold)),\n                auc_against_cross_impression_genuine=auc_against(positive, values)\n                    if pool.endswith(\'impostor\') else None, **describe_scores(values)))\n    return pd.DataFrame(summaries)\n'
exec(compile(AUDIT_SOURCE, '<matcher-failure-audit>', 'exec'))


In [ ]:

verify_score_metadata(calibration, partition, 'calibration')
verify_score_metadata(gate, partition, 'gate')
thresholds, replay_metrics, replay_summary = matcher.gate_analysis(calibration, gate)
if saved_thresholds != {f'{dataset}_{db}': value for (dataset, db), value in thresholds.items()}:
    raise RuntimeError('Exact CSV replay changed original calibration thresholds.')
if replay_summary != saved_gate:
    raise RuntimeError('Saved gate summary differs from exact saved-score replay.')
prior_metrics = pd.read_csv(MATCHER_DIR / 'gate_metrics.csv', float_precision='round_trip')
pd.testing.assert_frame_equal(replay_metrics, prior_metrics, check_dtype=False, check_exact=True)
print('Exact saved-score replay verified; gate UNCHANGED:')
print(json.dumps(replay_summary, indent=2))
print('Saved fixed-budget learning/runtime/memory history:')
print(saved_history.to_string(index=False))


In [ ]:

audit_config = dict(version='risk_matcher_failure_v1', matcher_configuration_sha256=pointer['configuration_sha256'],
    matcher_completion_sha256=pointer['completion_sha256'], checkpoint=latest,
    source_sha256=hashlib.sha256(AUDIT_SOURCE.encode()).hexdigest(),
    cpu_torch_version=str(torch.__version__), cpu_threads=torch.get_num_threads(),
    panels='all original gate groups; first8 fit groups/DB by fixed hash; first3 distinct impressions',
    epoch=0, crop_unchanged=True, thresholds='original29 calibration only',
    source_files={'restoration_baseline.py': file_digest(M0_DIR / 'restoration_baseline.py'),
        'fingerprint_degradation.py': file_digest(SIM_DIR / 'fingerprint_degradation.py')},
    inference_only=True, permits_restoration_training=False)
audit_sha = hashlib.sha256(json_bytes(audit_config)).hexdigest()
AUDIT_DIR = PROJECT_ROOT / 'diagnostics' / 'risk_matcher_failure_v1' / audit_sha[:20]
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
for name, data in {'configuration.json': json_bytes(audit_config), 'audit_helpers.py.txt': AUDIT_SOURCE.encode(),
        'gate_replay.json': json_bytes(replay_summary)}.items():
    path = AUDIT_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen audit differs: {path}')
    if not path.exists():
        write_verified(path, data)


In [ ]:

started = time.monotonic()
score_rows, crop_rows, preview_records = [], [], []
for split in ('fit', 'gate'):
    galleries, probes = panel_records(partition, split)
    gallery_features = embed_panel(galleries, ('identity', 'none'))
    clean_features = embed_panel(probes, ('identity', 'none'))
    for family, severity in [('identity', 'none')] + [('dry_like', level) for level in ('mild', 'moderate', 'severe')]:
        condition = f'{family}/{severity}'
        probe_features = clean_features if family == 'identity' else embed_panel(probes, (family, severity))
        matrix = (probe_features @ gallery_features.T).numpy()
        aligned = (probe_features * clean_features).sum(1).numpy()
        for i, probe in enumerate(probes):
            common = dict(inner_split=split, dataset=probe['dataset'], db=probe['db'],
                condition=condition, probe=probe['relative_path'])
            score_rows.append(dict(**common, gallery=probe['relative_path'],
                pool='same_source_control', score=float(aligned[i])))
            for j, gallery in enumerate(galleries):
                same_db = (probe['dataset'], probe['db']) == (gallery['dataset'], gallery['db'])
                if probe['identity_key'] == gallery['identity_key']:
                    pool = 'different_impression_genuine'
                elif probe['connected_group'] == gallery['connected_group']:
                    continue
                else:
                    pool = 'same_db_impostor' if same_db else 'cross_db_impostor'
                score_rows.append(dict(**common, gallery=gallery['relative_path'], pool=pool,
                    score=float(matrix[i, j])))
        print(f'CPU panel {split}/{condition}: {len(probes)} probes; no updates.', flush=True)
    unique = {row['relative_path']: row for row in galleries + probes}
    for record in unique.values():
        metadata, _, _ = crop_details(record)
        crop_rows.append(dict(inner_split=split, **metadata))
    if split == 'gate':
        for dataset, db in sorted({(row['dataset'], row['db']) for row in galleries}):
            gallery = next(row for row in galleries if (row['dataset'], row['db']) == (dataset, db))
            matching = [row for row in probes if row['identity_key'] == gallery['identity_key']]
            if len(matching) != 2:
                raise RuntimeError('Expected two distinct impressions for each fixed preview identity.')
            preview_records.append([gallery, *matching])
scores = pd.DataFrame(score_rows)
summary = summarize_panel(scores)
gate_cpu = scores[scores.inner_split.eq('gate') & scores.pool.isin(
    ['different_impression_genuine', 'same_db_impostor'])].copy()
gate_cpu['label'] = gate_cpu.pool.eq('different_impression_genuine').astype(int)
keys = ['dataset', 'db', 'condition', 'probe', 'gallery', 'label']
cpu_comparison = gate.merge(gate_cpu[keys + ['score']], on=keys, suffixes=('_saved_gpu', '_cpu'), validate='one_to_one')
if len(cpu_comparison) != len(gate) or len(cpu_comparison) != len(gate_cpu):
    raise RuntimeError('CPU gate re-inference does not cover every saved pair.')
cpu_comparison['absolute_score_difference'] = (
    cpu_comparison.score_saved_gpu - cpu_comparison.score_cpu).abs()
cutoffs = np.array([saved_thresholds[f'{row.dataset}_{row.db}'] for row in cpu_comparison.itertuples()])
cpu_comparison['decision_changed'] = (
    (cpu_comparison.score_saved_gpu >= cutoffs) != (cpu_comparison.score_cpu >= cutoffs))
cpu_drift = dict(max_absolute_score_difference=float(cpu_comparison.absolute_score_difference.max()),
    decision_changes=int(cpu_comparison.decision_changed.sum()), pairs=len(cpu_comparison),
    interpretation='Original saved GPU gate is authoritative; CPU re-inference never reopens training.')
for name, frame in [('panel_scores.csv', scores), ('panel_summary.csv', summary),
        ('crop_geometry.csv', pd.DataFrame(crop_rows)), ('cpu_vs_saved_gate.csv', cpu_comparison),
        ('saved_training_history.csv', saved_history)]:
    write_csv(AUDIT_DIR / name, frame)
write_verified(AUDIT_DIR / 'cpu_drift.json', json_bytes(cpu_drift))
print(json.dumps(cpu_drift, indent=2))
print(summary.groupby(['inner_split', 'condition', 'pool'], sort=True).agg(
    databases=('db', 'size'), macro_score=('mean', 'mean'),
    macro_acceptance=('acceptance_fraction', 'mean'),
    macro_auc=('auc_against_cross_impression_genuine', 'mean')).to_string())


In [ ]:

from matplotlib.patches import Rectangle
figure, axes = plt.subplots(len(preview_records), 6, figsize=(18, 3 * len(preview_records)), squeeze=False)
for row, records in enumerate(preview_records):
    for index, record in enumerate(records):
        metadata, reference, patch = crop_details(record)
        full_axis, patch_axis = axes[row, 2 * index:2 * index + 2]
        full_axis.imshow(reference, cmap='gray', vmin=0, vmax=1)
        full_axis.add_patch(Rectangle((metadata['left'], metadata['top']), SETTINGS['patch_size'],
            SETTINGS['patch_size'], fill=False, edgecolor='red', linewidth=1))
        full_axis.set_title(f"{record['dataset']}/{record['db']} | impression{index} | original", fontsize=8)
        patch_axis.imshow(patch, cmap='gray', vmin=0, vmax=1)
        patch_axis.set_title('Frozen crop (not anatomically registered)', fontsize=8)
        full_axis.axis('off')
        patch_axis.axis('off')
figure.tight_layout()
figure.savefig(AUDIT_DIR / 'fixed_gate_crop_preview.png', dpi=120)
plt.show()
plt.close(figure)


In [ ]:

if any(not torch.equal(value, model.state_dict()[key]) for key, value in saved_state.items()):
    raise RuntimeError('Read-only audit changed matcher weights.')
if any(parameter.requires_grad or parameter.grad is not None for parameter in model.parameters()):
    raise RuntimeError('Read-only audit unexpectedly enabled training or gradients.')
if file_digest(MATCHER_DIR / 'completion.json') != pointer['completion_sha256']:
    raise RuntimeError('Original29 completion changed during audit.')
verify_files(MATCHER_DIR, completion['file_sha256'])
files = ['configuration.json', 'audit_helpers.py.txt', 'gate_replay.json', 'panel_scores.csv',
    'panel_summary.csv', 'crop_geometry.csv', 'cpu_vs_saved_gate.csv', 'saved_training_history.csv',
    'cpu_drift.json', 'fixed_gate_crop_preview.png']
marker = dict(status='risk_matcher_failure_audit_complete', configuration_sha256=audit_sha,
    file_sha256={name: file_digest(AUDIT_DIR / name) for name in files},
    original29_gate_passed=saved_gate['passed'], permits_restoration_training=False,
    seconds=time.monotonic() - started, images_read='main TRAIN only',
    caveats=['No training or alternate checkpoint; no gate modification.',
        'No cross-impression anatomical overlap estimate without registration.',
        'Cross-DB easier negatives do not alone prove sensor shortcut.',
        'Fit versus gate is descriptive; no confidence or causal guarantee.'])
write_verified(AUDIT_DIR / 'completion.json', json_bytes(marker))
print('FAILED MATCHER AUDIT COMPLETE:', AUDIT_DIR)
print('Send panel_summary, CPU drift and fixed crop preview.30/31 REMAIN BLOCKED; no more29 training.')


## Public replacement candidates checked 2026-11-16 (not installed or loaded)

| Candidate | Verified evidence | Unresolved before use |
|---|---|---|
| [DMD / DMD++, official](https://github.com/Yu-Yy/DMD) | README links pretrained weights and states NIST SD14 training. The DMD Tsinghua link returned "Link does not exist"; its [official Drive backup page](https://drive.google.com/file/d/1iu8pbI2EjSkwOYsYczosJJDEhd1-JyzU/view) exposes `best_model.pth.tar`. [IJCB2024 paper](https://arxiv.org/html/2405.01199v2) uses minutia-centered, orientation-aligned128px patches and DPF segmentation masks for augmentation. | Need exact checkpoint checksum/training lineage, actual download and safe-load test; anchors and orientation convention; differentiable tensor preprocessing/scoring. Existing pipeline uses external minutiae and discrete matching. Not a drop-in192px cosine matcher. LICENSE contains Apache2 while README additionally says academic/noncommercial; clarify these terms before redistribution/use. |
| [DeepPrint reimplementation, BIOSIG2023 authors](https://github.com/tim-rohwedder/fixed-length-fingerprint-extractors) | LGPL3 code, public512-D Drive weight link (folder listing accessible). README describes SFinGe/MCYT training and says FVC2004/SD4 were not used in that paper. | Exact linked checkpoint training lineage/variant not attested by filename; paper-level data statement is not proof of weight-level nonoverlap.299px preprocessing, safe-load and input-gradient checks still needed. This is not the original DeepPrint authors' checkpoint. |
| [FLARE / FDD, official TIFS2026](https://github.com/Yu-Yy/FLARE) | Public FDD/pose weight links; official extraction uses pose-aligned input. | Full checkpoint/pretraining provenance not established here; inference entry point uses no_grad and NumPy scoring, so a tensor gradient path needs verification. LICENSE contains Apache2 while README additionally says academic/noncommercial; do not silently assume unrestricted licensing. |

**Preferred next candidate for an isolated feasibility check: DMD**, because official weights are explicitly
described as SD14-trained and its local alignment addresses the weakness of arbitrary image-coordinate crops.
This is NOT approval to swap it into30. Public links do not prove downloadability, safe loading,
input-gradient support, usable quality, or absence of all identity overlap.
Use NBIS-derived, frozen train-only anchors only after coordinate/orientation conversion is verified;
do not use proprietary VeriFinger as an unstated dependency. Any feature loss would initially be an
adaptation using fixed anchors, not a fully differentiable reproduction of the complete DMD matcher.

No automatic weight download, unsafe pickle loading, external biometric upload, validation/test fitting,
new loss experiment, or resource request occurs in this notebook.
